# 1-2절 연습 문제 풀이

이 노트북은 1-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch01/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 1-7

[코드 1-29]에서 1일 총비용 계산의 연산 그래프를 종이에 그려 보자.

### 풀이

[코드 1-29]의 계산을 연산 그래프로 나타내면 다음과 같다. 잎(leaf)에 해당하는 `unit_costs`가 `requires_grad=True`인 텐서이고, 화살표를 따라 거슬러 올라가며 기울기가 계산된다.

```
unit_costs (5,)  daily_usage (5,)
      \               /
       \             /
        (요소별 곱)  ->  daily_costs (5,)
                              |
                            sum()
                              |
                        total_cost (스칼라)
```

- `daily_costs = unit_costs * daily_usage`: 요소별 곱셈 노드
- `total_cost = daily_costs.sum()`: 합계 노드

역전파는 `total_cost`에서 출발한다. 합계 노드는 기울기를 그대로 흘려보내므로 각 `daily_costs` 요소의 기울기는 1이고, 곱셈 노드에서는 상대 피연산자가 그대로 기울기가 되어 `unit_costs.grad`는 `daily_usage`와 같아진다.

In [ ]:
# 그래프 구조를 코드로 확인해 본다.
unit_costs = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], requires_grad=True)
daily_usage = torch.tensor([10.0, 5.0, 2.0, 8.0, 1.0])

daily_costs = unit_costs * daily_usage
total_cost = daily_costs.sum()
total_cost.backward()

print(f'total_cost를 만든 연산: {total_cost.grad_fn}')
print(f'그 이전 연산:          {total_cost.grad_fn.next_functions[0][0]}')
print(f'unit_costs.grad: {unit_costs.grad}')
print(f'daily_usage:     {daily_usage}')
print('-> 두 값이 같으므로 곱셈 노드의 기울기는 상대 피연산자임을 알 수 있다.')

## 연습 1-8

[도전 문제] 카페 주인과 재료 공급업자는 모든 재료의 단위가격을 한 번에 낮추기 어렵다고 판단했다. 대신 한 번 최적화할 때마다 영향력이 가장 큰 두 재료의 단위가격만 낮추기로 했다. 이 방식으로 20회 반복해 최적화하는 과정을 구현해 보자.

힌트: 각 반복에서 기울기를 구한 뒤 다음 반복 전에 이전 기울기를 초기화하므로, 그 사이에는 grad 속성의 값을 자유롭게 바꿔도 된다.

In [ ]:
# 영향력(기울기)이 가장 큰 두 재료의 단위가격만 20회 반복해 낮춘다.
unit_costs = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], requires_grad=True)
daily_usage = torch.tensor([10.0, 5.0, 2.0, 8.0, 1.0])
LR = 0.01
TOP_K = 2

for step in range(20):
    total_cost = (unit_costs * daily_usage).sum()
    total_cost.backward()

    with torch.no_grad():
        # 기울기가 큰 상위 두 개의 위치만 남기고 나머지 기울기는 0으로 만든다.
        top_idx = unit_costs.grad.abs().topk(TOP_K).indices
        mask = torch.zeros_like(unit_costs.grad)
        mask[top_idx] = 1.0
        unit_costs.grad *= mask          # 선택되지 않은 재료는 갱신되지 않는다.
        unit_costs -= LR * unit_costs.grad
        unit_costs.clamp_(min=0.0)       # 단위가격이 음수가 되지 않도록 제한

    if step % 5 == 0 or step == 19:
        print(f'{step + 1:2d}회 - 총비용 {total_cost.item():6.2f} / '
              f'선택된 재료 {top_idx.tolist()}')
    unit_costs.grad.zero_()

print(f'\n최종 단위가격: {[round(v, 3) for v in unit_costs.tolist()]}')

기울기의 절댓값이 큰 두 위치만 남기는 마스크를 만들어 `grad`에 곱하면, 나머지 재료의 기울기가 0이 되어 갱신에서 제외된다. 힌트대로 `zero_()`로 초기화하기 전이라면 `grad` 값을 직접 바꿔도 다음 반복에 영향을 주지 않는다.

여기서는 사용량(`daily_usage`)이 많은 재료의 기울기가 항상 크므로 매 반복 같은 두 재료가 선택된다.